# 03 — Pipeline Completo del Router MoE (Pasos A → M)

Este notebook implementa el **pipeline completo** del Vision Router para el sistema MoE,
siguiendo exactamente el diagrama propuesto por el profesor:

```
A → Datos crudos (5 datasets)
B → DataLoader Mixto
C → Collate Personalizado (separa 2D / 3D)
D,E → Patch Embedding (2D: 16×16 | 3D: 8×8×8)
F,G → Proyección Lineal compartida → 192 dim
H → Padding de secuencia → tensor homogéneo [B, 513, 192]
I → Token CLS + Positional Embeddings
J → 12 Bloques de Self-Attention
K → Extracción del Token CLS Final
L → Vector de Representación Global: 1×192
M → Capa Lineal del Router: 192 → 5
```

**Nota técnica clave:** El `SwitchablePatchEmbed` usa *corrimiento de ventana*:
no es un modelo, es un **método** que convierte imágenes/volúmenes en valores
característicos de dimensión homogénea `1×192`, sin importar si la entrada es 2D o 3D.

Al final, los CLS tokens se guardan en disco para el **Estudio de Ablación** posterior
(Linear vs GMM vs Naive Bayes vs k-NN).


## Fase 0: Entorno y Configuración (Punto A)
Instalamos dependencias, montamos Drive y configuramos rutas.


In [1]:
!pip install -q SimpleITK nibabel torch torchvision monai einops timm
from google.colab import drive
import os, time, shutil, glob, zipfile, subprocess
from pathlib import Path
from collections import Counter
import numpy as np
import cv2
from PIL import Image
import SimpleITK as sitk
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from monai.networks.blocks import PatchEmbed
import random
from torch.cuda.amp import autocast
from sklearn.model_selection import train_test_split
import timm

# ---------- Montar Google Drive ----------
drive.mount('/content/drive')

# ---------- Rutas ----------
RAW_DIR    = '/content/drive/MyDrive/PROYECTO_MOE_VISION/01_Data/Raw/'
LOCAL_DEST = '/content/datasets/'
PROC_BASE  = '/content/drive/MyDrive/PROYECTO_MOE_VISION/02_Data_Processed/'
FEATURE_DIR = '/content/drive/MyDrive/PROYECTO_MOE_VISION/03_CLS_Features/'

os.makedirs(LOCAL_DEST, exist_ok=True)
os.makedirs(PROC_BASE, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

# ---------- Mapeo de datasets (consigna) ----------
DATASET_ROOTS = {
    'NIH':      (os.path.join(LOCAL_DEST, 'NIH Chest X ray 14'),                0),
    'ISIC':     (os.path.join(LOCAL_DEST, 'ISIC 2019'),                         1),
    'Osteo':    (os.path.join(LOCAL_DEST, 'Knee Osteoarthritis Classification'), 2),
    'LUNA16':   (os.path.join(LOCAL_DEST, 'Luna16 Lung Cancer Dataset'),         3),
    'Pancreas': (os.path.join(LOCAL_DEST, 'Pancreas Cancer'),                    4),
}

print('Rutas configuradas.')
print(f'  RAW_DIR:     {RAW_DIR}')
print(f'  LOCAL_DEST:  {LOCAL_DEST}')
print(f'  FEATURE_DIR: {FEATURE_DIR}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 87.7 MB/s eta 0:00:00


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Mounted at /content/drive
Rutas configuradas.
  RAW_DIR:     /content/drive/MyDrive/PROYECTO_MOE_VISION/01_Data/Raw/
  LOCAL_DEST:  /content/datasets/
  FEATURE_DIR: /content/drive/MyDrive/PROYECTO_MOE_VISION/03_CLS_Features/


## Fase 1: Extracción de datos a disco local


In [2]:
def extract_datasets_colab(raw_dir=RAW_DIR, local_dest=LOCAL_DEST):
    """Copia ZIPs de Drive a /content/datasets/ y los descomprime."""
    if not os.path.exists(raw_dir):
        print(f'Ruta {raw_dir} no existe. No se extrae nada.')
        return
    zip_files = sorted([f for f in os.listdir(raw_dir) if f.lower().endswith('.zip')])
    print(f'Encontrados {len(zip_files)} zips.')
    for zip_name in zip_files:
        print('=' * 60)
        print(f'Procesando: {zip_name}')
        drive_zip_path = os.path.join(raw_dir, zip_name)
        dataset_name = os.path.splitext(zip_name)[0]
        unzip_dir = os.path.join(local_dest, dataset_name)
        local_zip_path = os.path.join(local_dest, zip_name)
        if os.path.isdir(unzip_dir) and len(os.listdir(unzip_dir)) > 0:
            print(f' Ya existe: {unzip_dir} (omitido).')
            continue
        print(' 1. Copiando ZIP...')
        shutil.copy2(drive_zip_path, local_zip_path)
        os.makedirs(unzip_dir, exist_ok=True)
        print(f' 2. Descomprimiendo en {unzip_dir}...')
        subprocess.run(['unzip', '-q', local_zip_path, '-d', unzip_dir], check=True)
        print(' 3. Borrando ZIP local.')
        os.remove(local_zip_path)
        # ZIPs internos
        for iz in glob.glob(os.path.join(unzip_dir, '**', '*.zip'), recursive=True):
            print(f' -> ZIP interno: {iz}')
            subprocess.run(['unzip', '-q', iz, '-d', os.path.dirname(iz)], check=True)
            os.remove(iz)
    print('\nExtracción completa.')

# Ejecutar una sola vez por sesión de Colab
extract_datasets_colab()


Encontrados 5 zips.
Procesando: ISIC 2019.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/ISIC 2019...
 3. Borrando ZIP local.
Procesando: Knee Osteoarthritis Classification.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/Knee Osteoarthritis Classification...
 3. Borrando ZIP local.
Procesando: Luna16 Lung Cancer Dataset.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/Luna16 Lung Cancer Dataset...
 3. Borrando ZIP local.
Procesando: NIH Chest X ray 14.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/NIH Chest X ray 14...
 3. Borrando ZIP local.
Procesando: Pancreas Cancer.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/Pancreas Cancer...
 3. Borrando ZIP local.
 -> ZIP interno: /content/datasets/Pancreas Cancer/batch_1.zip

Extracción completa.


## Fase 2: AdaptivePreprocessor (Punto A — Carga nativa)
Carga archivos PNG/JPG/MHD/NIfTI y los convierte en tensores nativos:
- 2D: `[3, 224, 224]`
- 3D: `[1, 64, 64, 64]`

**Sin metadatos.** Solo mira la forma del tensor (`ndim`) para saber si es 2D o 3D.


In [59]:
class AdaptivePreprocessor:
    """Carga archivos y retorna tensores nativos (2D o 3D)."""
    def __init__(self, size_2d=(224, 224), size_3d=(64, 64, 64), hu_window_ct=(-1000, 400)):
        self.size_2d = size_2d
        self.size_3d = size_3d
        self.hu_min, self.hu_max = hu_window_ct

    def __call__(self, file_path):
        ext = file_path.lower()
        if ext.endswith(('.png', '.jpg', '.jpeg')): return self._process_2d(file_path)
        if ext.endswith('.mha'):                     return self._process_mha(file_path)
        if ext.endswith(('.mhd', '.nii.gz', '.nii')): return self._process_3d(file_path)
        raise ValueError(f'Formato no soportado: {file_path}')

    def _process_2d(self, path):
        img = Image.open(path).convert('RGB')
        transform = T.Compose([T.Resize(self.size_2d), T.ToTensor()])
        return transform(img)

    def _process_mha(self, path):
        itk_img = sitk.ReadImage(path)
        size = itk_img.GetSize()
        arr = sitk.GetArrayFromImage(itk_img).astype(np.float32)
        if len(size) == 2:                       return self._array_2d_to_tensor(arr)
        if len(size) == 3 and size[2] <= 1:
            arr = np.squeeze(arr)
            if arr.ndim == 2:                    return self._array_2d_to_tensor(arr)
        return self._volume_array_to_tensor(arr, path)

    def _array_2d_to_tensor(self, arr):
        if arr.max() > 1.5:
            arr = arr / 255.0 if arr.max() > 2.0 else (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
        t = torch.from_numpy(arr).unsqueeze(0)
        if t.shape[1:] != torch.Size(self.size_2d):
            t = F.interpolate(t.unsqueeze(0), size=self.size_2d, mode='bilinear', align_corners=False).squeeze(0)
        if t.shape[0] == 1:  t = t.repeat(3, 1, 1)
        elif t.shape[0] > 3: t = t[:3]
        return t

    def _volume_array_to_tensor(self, img_arr, path_hint=''):
        amin, amax = float(np.nanmin(img_arr)), float(np.nanmax(img_arr))
        pre_norm = amax <= 1.5 and amin >= -1e-2 and 'Pancreas' in str(path_hint).replace('\\', '/')
        if not pre_norm:
            img_arr = np.clip(img_arr, self.hu_min, self.hu_max)
            img_arr = (img_arr - self.hu_min) / (self.hu_max - self.hu_min + 1e-8)
        else:
            img_arr = np.clip(img_arr, 0.0, 1.0)
        t = torch.tensor(img_arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=self.size_3d, mode='trilinear', align_corners=False)
        return t.squeeze(0)  # [1, 64, 64, 64]

    def _process_3d(self, path):
        if path.endswith('.mhd'):
            itk_img = sitk.ReadImage(path)
            img_arr = sitk.GetArrayFromImage(itk_img).astype(np.float32)
        else:
            nii_img = nib.load(path)
            img_arr = np.transpose(nii_img.get_fdata().astype(np.float32), (2, 1, 0))
        return self._volume_array_to_tensor(img_arr, path_hint=path)

preprocessor = AdaptivePreprocessor()
print('AdaptivePreprocessor listo.')


AdaptivePreprocessor listo.


## Fase 3: Escáner de archivos
Función que recorre recursivamente un directorio y retorna los paths de los archivos médicos.


In [60]:
VALID_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.mha', '.mhd', '.nii', '.nii.gz')

def scan_dataset_files(root_path, max_files=None):
    """Escanea recursivamente y retorna paths de archivos médicos válidos."""
    files = []
    for dirpath, _, filenames in os.walk(root_path):
        for fname in sorted(filenames):
            if fname.lower().endswith(VALID_EXTENSIONS):
                files.append(os.path.join(dirpath, fname))
    if max_files:
        files = files[:max_files]
    return files

# Verificación rápida
for name, (path, did) in DATASET_ROOTS.items():
    if os.path.exists(path):
        count = len(scan_dataset_files(path, max_files=None))
        print(f'  {name} (ID={did}): {count} archivos en {path}')
    else:
        print(f'  {name} (ID={did}): RUTA NO ENCONTRADA — {path}')


  NIH (ID=0): 112120 archivos en /content/datasets/NIH Chest X ray 14
  ISIC (ID=1): 25331 archivos en /content/datasets/ISIC 2019
  Osteo (ID=2): 12712 archivos en /content/datasets/Knee Osteoarthritis Classification
  LUNA16 (ID=3): 889 archivos en /content/datasets/Luna16 Lung Cancer Dataset
  Pancreas (ID=4): 557 archivos en /content/datasets/Pancreas Cancer


## Fase 4: DataLoader Mixto (Puntos B y C)

### B: `MixedMedicalDataset`
Dataset unificado que mezcla los 5 datasets. Retorna `(tensor, dataset_id)`.

### C: `mixed_collate_fn`
Collate personalizado que **NO** intenta apilar tensores de distinta forma.
Devuelve listas crudas para que el modelo procese cada muestra individualmente.


In [61]:
class MixedMedicalDataset(torch.utils.data.Dataset):
    """Dataset unificado: mezcla imágenes 2D y volúmenes 3D de los 5 datasets."""
    def __init__(self, file_list, dataset_ids, preprocessor):
        self.file_list = file_list
        self.dataset_ids = dataset_ids
        self.preprocessor = preprocessor

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        tensor = self.preprocessor(self.file_list[idx])
        return tensor, self.dataset_ids[idx]


def mixed_collate_fn(batch):
    """
    Collate personalizado (Punto C).
    NO apila tensores — devuelve listas crudas.
    Esto es necesario porque 2D=[3,224,224] y 3D=[1,64,64,64] tienen formas distintas.
    """
    tensors = [b[0] for b in batch]
    dataset_ids = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return tensors, dataset_ids


def build_router_dataloader(roots, preprocessor, batch_size=8, num_workers=2, sample_cap=1000):
    """
    Construye el DataLoader Mixto con BALANCEO (Punto B).
    roots: dict {nombre: (path, dataset_id)}
    sample_cap: int. Límite máximo de archivos por dataset para evitar desbalance.
    """
    all_files, all_ids = [], []
    for name, (path, did) in roots.items():
        files = scan_dataset_files(path)
        # --- LÓGICA DE BALANCEO ---
        if len(files) > sample_cap:
            random.seed(42)
            files = random.sample(files, sample_cap)
            print(f'  [{name}] Balanceado: reducido a {sample_cap} muestras (ID={did})')
        else:
            print(f'  [{name}] {len(files)} archivos cargados en total (ID={did})')
        all_files.extend(files)
        all_ids.extend([did] * len(files))

    dataset = MixedMedicalDataset(all_files, all_ids, preprocessor)
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size,
        shuffle=True,
        collate_fn=mixed_collate_fn,
        num_workers=num_workers,
        pin_memory=True
    )
    print(f'\nDataLoader listo: {len(dataset)} muestras, batch_size={batch_size}')
    return loader

print('MixedMedicalDataset y mixed_collate_fn definidos.')


MixedMedicalDataset y mixed_collate_fn definidos.


## Fase 5: SwitchablePatchEmbed (Puntos D → I)

Este módulo implementa el **corrimiento de ventana** que el profesor indicó:
- **D**: Foto 2D → parches de 16×16
- **E**: Volumen 3D → cubitos de 8×8×8
- **F, G**: Proyección lineal → cada parche se convierte en un vector de **192 dimensiones**
- **H**: Padding de secuencia (todos los batches al mismo largo)
- **I**: Se agrega Token CLS + Positional Embeddings

**Resultado:** Tensor homogéneo `[B, 513, 192]` listo para el Transformer.


In [79]:
from monai.networks.blocks import PatchEmbed
import torch
import torch.nn as nn

class SwitchablePatchEmbed(nn.Module):
    """
    Switchable Patch Embedding (SPE) — Pasos D→I.
    Versión corregida y más robusta.
    """
    def __init__(self, embed_dim=192, patch_size_2d=16, patch_size_3d=8, in_channels_2d=3):
        super().__init__()
        self.embed_dim = embed_dim

        # D: Patch Embedding 2D
        self.patch_embed_2d = PatchEmbed(
            spatial_dims=2,
            in_chans=in_channels_2d,      # ← Correcto: in_chans
            patch_size=patch_size_2d,
            embed_dim=embed_dim           # ← Correcto: embed_dim
        )

        # E: Patch Embedding 3D
        self.patch_embed_3d = PatchEmbed(
            spatial_dims=3,
            in_chans=1,
            patch_size=patch_size_3d,
            embed_dim=embed_dim
        )

        # 1 (CLS) + max(14*14, 8**3) = 1 + 512 = 513 tokens
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 513, embed_dim))

    def _patch_tokens_to_sequence(self, patch_tokens: torch.Tensor) -> torch.Tensor:
        """MONAI puede devolver [B,N,D] o tensores espaciales [B,H',W',D] / [B,D,H',W']."""
        x = patch_tokens
        if x.dim() == 3 and x.size(-1) == self.embed_dim:
            return x.squeeze(0) if x.size(0) == 1 else x.reshape(-1, self.embed_dim)
        if x.dim() >= 3 and x.size(-1) == self.embed_dim:
            return x.reshape(-1, self.embed_dim)
        if x.dim() == 4 and x.size(1) == self.embed_dim:
            x = x.flatten(2).transpose(1, 2).contiguous()
            return x.reshape(-1, self.embed_dim)
        if x.dim() == 5 and x.size(1) == self.embed_dim:
            x = x.flatten(2).transpose(1, 2).contiguous()
            return x.reshape(-1, self.embed_dim)
        raise RuntimeError(f'Forma de patch tokens no soportada: {tuple(patch_tokens.shape)}')

    def forward(self, batch_tensors):
        batch_size = len(batch_tensors)
        tokens_list = []

        for sample in batch_tensors:
            sample = sample.unsqueeze(0)  # [1, C, ...]

            if sample.ndim == 4:
                if sample.shape[1] == 1:
                    sample = sample.repeat(1, 3, 1, 1)
                patch_tokens = self.patch_embed_2d(sample)
            elif sample.ndim == 5:
                patch_tokens = self.patch_embed_3d(sample)
            else:
                raise ValueError(f'Tensor de entrada inválido (ndim={sample.ndim}): {tuple(sample.shape)}')

            seq = self._patch_tokens_to_sequence(patch_tokens)
            tokens_list.append(seq)

        # H: Padding
        padded = torch.nn.utils.rnn.pad_sequence(tokens_list, batch_first=True)

        # Máscara (True = válido)
        lengths = torch.tensor([t.size(0) for t in tokens_list], device=padded.device)
        max_len = padded.size(1)
        mask = torch.arange(max_len, device=padded.device)[None, :] < lengths[:, None]

        # I: CLS + Positional
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        final_tokens = torch.cat((cls_tokens, padded), dim=1)

        cls_mask = torch.ones(batch_size, 1, dtype=torch.bool, device=mask.device)
        final_mask = torch.cat((cls_mask, mask), dim=1)

        final_tokens = final_tokens + self.pos_embed[:, :final_tokens.size(1), :]

        return final_tokens, final_mask

## Fase 6: VisionRouter Completo (Puntos J → M)

El modelo completo que orquesta todo:
- **J**: 12 bloques de Self-Attention (`TransformerEncoder`)
- **K**: Extracción del Token CLS final (posición 0)
- **L**: Vector de representación global `1×192`
- **M**: Capa lineal `192 → 5` para decidir a qué experto enviar

El modelo retorna `(logits, cls_vector)` para poder:
1. Entrenar el router con CrossEntropyLoss
2. Extraer los CLS para el ablation study


In [80]:
class VisionRouter(nn.Module):
    def __init__(self, embed_dim=192, num_experts=5, num_layers=12, pretrained=True):
        super().__init__()

        self.patch_embed = SwitchablePatchEmbed(embed_dim=embed_dim)

        # Usamos ViT-Tiny de timm (mucho más optimizado)
        self.vit = timm.create_model(
            'vit_tiny_patch16_224',
            pretrained=pretrained,
            num_classes=0,           # sin cabeza de clasificación
            global_pool='',          # no pooling
            img_size=224,            # solo referencia, no lo usamos realmente
        )

        # Reemplazamos el patch_embed original de timm por nuestro Switchable
        self.vit.patch_embed = nn.Identity()

        # Si queremos controlar el número de layers (opcional)
        if num_layers < 12:
            self.vit.blocks = self.vit.blocks[:num_layers]

        # Cabeza del router
        self.router_head = nn.Linear(embed_dim, num_experts)

    def forward(self, batch_tensors):
        # A → I: Patch + CLS + Positional
        x, mask = self.patch_embed(batch_tensors)   # [B, seq_len+1, 192]

        # timm ViT espera entrada sin CLS token + positional ya incluido
        # Como nosotros ya agregamos CLS y positional, pasamos directamente

        # Opción más limpia: forward manual solo de los bloques
        for blk in self.vit.blocks:
            x = blk(x)

        # K/L: Extraer CLS token (posición 0)
        cls_token = x[:, 0]                    # [B, 192]

        # M: Router head
        logits = self.router_head(cls_token)   # [B, 5]

        return logits, cls_token

print('VisionRouter definido (Pasos J→M).')


VisionRouter definido (Pasos J→M).


## Fase 7: Prueba con datos dummy (sin disco)
Verificamos que el pipeline completo funciona antes de usar datos reales.


In [90]:
# ---------- Prueba rápida ----------
model = VisionRouter(embed_dim=192, num_experts=5, num_layers=12, pretrained=False) # Pretrained False por velocidad en prueba
model.eval()

# Simulando un batch mixto (salida del mixed_collate_fn):
dummy_batch = [
    torch.randn(3, 224, 224),   # Imagen 2D (NIH, ISIC, Osteo)
    torch.randn(1, 64, 64, 64), # Volumen 3D (LUNA16, Pancreas)
    torch.randn(3, 224, 224),   # Imagen 2D
]

with torch.no_grad():
    logits, cls_features = model(dummy_batch)

print(f'Logits shape:       {logits.shape}')        # → [3, 5]
print(f'CLS features shape: {cls_features.shape}')  # → [3, 192]
print(f'Decisiones:         {logits.argmax(dim=1)}') # → tensor de 3 IDs de experto
print()
print('Pipeline A→M funciona correctamente.')
print(f'   Cada muestra produce un vector 1×192 y una decisión entre {logits.shape[1]} expertos.')

del model  # liberar VRAM


Logits shape:       torch.Size([3, 5])
CLS features shape: torch.Size([3, 192])
Decisiones:         tensor([1, 1, 1])

Pipeline A→M funciona correctamente.
   Cada muestra produce un vector 1×192 y una decisión entre 5 expertos.


## Fase 8: Extracción de CLS Tokens a Disco

**Recomendación del profesor (Fase 0 del Ablation Study):**

Una vez que el backbone ViT esté entrenado (o cargado de `timm`), pasamos
**todo el dataset** por el Router y guardamos los vectores `1×192` en disco.

Esto permite hacer el estudio de ablación (Linear vs GMM vs NB vs k-NN)
en segundos, sin re-procesar las imágenes pesadas.


In [88]:
def extract_and_save_cls_features(dataloader, model, save_path, device='cpu'):
    """
    Fase 0 del Ablation Study:
    Extrae los vectores CLS (1×192) de todo el dataset y los guarda en disco.
    """
    model.to(device)
    model.eval()
    all_features = []
    all_labels = []

    print(f'Extrayendo CLS features → {save_path}')
    with torch.no_grad():
        with autocast():  # ⚡ Acelera el Transformer enormemente en GPU
            for i, (tensors, ids) in enumerate(dataloader):
                # Mover tensores al dispositivo
                tensors = [t.to(device) for t in tensors]
                _, cls_features = model(tensors)
                all_features.append(cls_features.cpu())
                all_labels.append(ids.cpu())
                if (i + 1) % 10 == 0:
                    print(f'  Procesados {i + 1} batches ...')

    features_array = torch.cat(all_features, dim=0).numpy()
    labels_array = torch.cat(all_labels, dim=0).numpy()

    print(f'✅ Extracción Completa: {features_array.shape[0]} muestras × {features_array.shape[1]} dims')
    print(f'   Distribución Total: {dict(Counter(labels_array))}')

    # --- Split 80/20 y guardado en formato NumPy (.npy) como pidió el profesor ---
    print('\nDividiendo en Train (80%) y Val (20%) y guardando como .npy...')
    Z_train, Z_val, y_train, y_val = train_test_split(
        features_array, labels_array, test_size=0.20, random_state=42, stratify=labels_array
    )

    os.makedirs(save_path, exist_ok=True)

    np.save(os.path.join(save_path, 'Z_train.npy'), Z_train)
    np.save(os.path.join(save_path, 'Z_val.npy'), Z_val)
    np.save(os.path.join(save_path, 'y_train_expert.npy'), y_train)
    np.save(os.path.join(save_path, 'y_val_expert.npy'), y_val)

    print(f'✅ Guardado en: {save_path}')
    print(f'   Train: Z={Z_train.shape}, y={dict(Counter(y_train))}')
    print(f'   Val:   Z={Z_val.shape}, y={dict(Counter(y_val))}')

    return Z_train, Z_val, y_train, y_val

print('extract_and_save_cls_features definida con división Train/Val.')


extract_and_save_cls_features definida con división Train/Val.


## Fase 9: Ejecución con datos reales

Construimos el DataLoader con los 5 datasets y ejecutamos la extracción.

> **Nota:** Para pruebas rápidas, usa `max_per_dataset=10`.
> Para la extracción final, pon `max_per_dataset=None`.


In [83]:
# ---------- Construir DataLoader Mixto ----------
# Para prueba rápida: max_per_dataset=10
# Para extracción completa: max_per_dataset=None

loader = build_router_dataloader(
    roots=DATASET_ROOTS,
    preprocessor=preprocessor,
    batch_size=16,      # ⚡ Batch más grande gracias a las optimizaciones
    num_workers=2,
    sample_cap=1000     # ← Balanceo robusto
)


  [NIH] Balanceado: reducido a 1000 muestras (ID=0)
  [ISIC] Balanceado: reducido a 1000 muestras (ID=1)
  [Osteo] Balanceado: reducido a 1000 muestras (ID=2)
  [LUNA16] 889 archivos cargados en total (ID=3)
  [Pancreas] 557 archivos cargados en total (ID=4)

DataLoader listo: 4446 muestras, batch_size=16


In [84]:
# ---------- Instanciar modelo y extraer CLS ----------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')

router = VisionRouter(embed_dim=192, num_experts=5, num_layers=12, pretrained=True)

SAVE_DIR = os.path.join(FEATURE_DIR, 'ablation_data_v1')

Z_train, Z_val, y_train, y_val = extract_and_save_cls_features(
    dataloader=loader,
    model=router,
    save_path=SAVE_DIR,
    device=device
)


Dispositivo: cuda
Extrayendo CLS features → /content/drive/MyDrive/PROYECTO_MOE_VISION/03_CLS_Features/ablation_data_v1


/tmp/ipykernel_3132/1993270240.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # ⚡ Acelera el Transformer enormemente en GPU


  Procesados 10 batches ...
  Procesados 20 batches ...
  Procesados 30 batches ...
  Procesados 40 batches ...
  Procesados 50 batches ...
  Procesados 60 batches ...
  Procesados 70 batches ...
  Procesados 80 batches ...
  Procesados 90 batches ...
  Procesados 100 batches ...
  Procesados 110 batches ...
  Procesados 120 batches ...
  Procesados 130 batches ...
  Procesados 140 batches ...
  Procesados 150 batches ...
  Procesados 160 batches ...
  Procesados 170 batches ...
  Procesados 180 batches ...
  Procesados 190 batches ...
  Procesados 200 batches ...
  Procesados 210 batches ...
  Procesados 220 batches ...
  Procesados 230 batches ...
  Procesados 240 batches ...
  Procesados 250 batches ...
  Procesados 260 batches ...
  Procesados 270 batches ...
✅ Extracción Completa: 4446 muestras × 192 dims
   Distribución Total: {np.int64(3): 889, np.int64(1): 1000, np.int64(4): 557, np.int64(0): 1000, np.int64(2): 1000}

Dividiendo en Train (80%) y Val (20%) y guardando como .npy.

## Fase 10: Estudio de Ablación (Routing Mechanisms)

Una vez guardados los CLS en disco, comparamos qué clasifica mejor el flujo:
1. **Linear + Softmax** (Capa densa, entrenada con gradiente)
2. **GMM** (Gaussian Mixture Model, sklearn)
3. **Naive Bayes** (GaussianNB, sklearn)
4. **k-NN** (FAISS, distancia coseno)

> La implementación detallada de cada mecanismo va en el notebook 04.


In [87]:
# ---------- Cargar CLS features guardados (Directo como pide el profesor) ----------
# Z_train = np.load(os.path.join(SAVE_DIR, 'Z_train.npy'))
# Z_val   = np.load(os.path.join(SAVE_DIR, 'Z_val.npy'))
# y_train_expert = np.load(os.path.join(SAVE_DIR, 'y_train_expert.npy'))
# y_val_expert   = np.load(os.path.join(SAVE_DIR, 'y_val_expert.npy'))
# print(f'Features cargados. Train: {Z_train.shape}, Val: {Z_val.shape}')

# ---------- Mecanismo A: Linear + Softmax ----------
# (Ya implementado dentro del VisionRouter como `router_head`, pero se entrenará con SGD)

# ---------- Mecanismo B: GMM ----------
# from sklearn.mixture import GaussianMixture
# gmm = GaussianMixture(n_components=5, covariance_type='full', max_iter=200, random_state=42)
# gmm.fit(Z_train)
# probs = gmm.predict_proba(Z_val)
# routing_pred_gmm = probs.argmax(axis=1)

# ---------- Mecanismo C: Naive Bayes ----------
# from sklearn.naive_bayes import GaussianNB
# nb = GaussianNB()
# nb.fit(Z_train, y_train_expert)
# probs = nb.predict_proba(Z_val)
# routing_pred_nb = probs.argmax(axis=1)

# ---------- Mecanismo D: k-NN (FAISS) ----------
# import faiss
# # faiss modifica el array inplace (necesita float32 copy)
# Z_train_f32 = Z_train.copy()
# Z_val_f32   = Z_val.copy()
# faiss.normalize_L2(Z_train_f32)
# faiss.normalize_L2(Z_val_f32)
# index = faiss.IndexFlatIP(Z_train_f32.shape[1])
# index.add(Z_train_f32)
# D, I = index.search(Z_val_f32, k=5)
# routing_pred_knn = np.apply_along_axis(
#     lambda x: np.bincount(x, minlength=5).argmax(), axis=1, arr=y_train_expert[I]
# )

print('Boilerplate del Ablation Study (Sckikit-Learn & FAISS) listo.')
print('Descomenta las secciones para ejecutar cada mecanismo.')


Boilerplate del Ablation Study (Sckikit-Learn & FAISS) listo.
Descomenta las secciones para ejecutar cada mecanismo.


In [86]:
# --- Fase 19: entrenar solo router_head (backbone congelado) ---

def _switch_aux_loss(router_probs: torch.Tensor, num_experts: int) -> torch.Tensor:
    """L_aux = N * sum_i f_i * P_i (consigna). router_probs: [B, N] softmax."""
    B = router_probs.size(0)
    if B == 0:
        return router_probs.sum() * 0.0
    hard = router_probs.argmax(dim=1)
    with torch.no_grad():
        f = torch.bincount(hard, minlength=num_experts).float() / float(B)
    P = router_probs.mean(dim=0)
    return num_experts * (f * P).sum()


def set_train_router_head_only(model: VisionRouter, head_only: bool = True):
    """Si head_only=True: congela patch_embed y vit; entrena solo router_head."""
    if head_only:
        for p in model.patch_embed.parameters():
            p.requires_grad = False
        for p in model.vit.parameters():
            p.requires_grad = False
        for p in model.router_head.parameters():
            p.requires_grad = True
    else:
        for p in model.parameters():
            p.requires_grad = True


def train_router_head_one_epoch(
    model: VisionRouter,
    dataloader,
    optimizer,
    device,
    *,
    alpha_aux: float = 0.05,
    scaler=None,
    max_batches: int | None = None,
):
    """Una época: CE al experto verdadero (dataset_id) + alpha * L_aux."""
    model.train()
    # Solo cabeza: el resto en eval (BatchNorm si hubiera)
    model.patch_embed.eval()
    model.vit.eval()
    model.router_head.train()

    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_ce = 0.0
    total_aux = 0.0
    n_samples = 0
    n_correct = 0
    use_amp = scaler is not None and str(device).startswith('cuda')

    for bi, (tensors, expert_ids) in enumerate(dataloader):
        if max_batches is not None and bi >= max_batches:
            break
        tensors = [t.to(device, non_blocking=True) for t in tensors]
        expert_ids = expert_ids.to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            logits, _ = model(tensors)
            probs = F.softmax(logits, dim=1)
            ce = criterion(logits, expert_ids)
            aux = _switch_aux_loss(probs, model.router_head.out_features)
            loss = ce + alpha_aux * aux

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        bs = expert_ids.size(0)
        total_loss += loss.item() * bs
        total_ce += ce.item() * bs
        total_aux += aux.item() * bs
        n_samples += bs
        n_correct += (logits.argmax(1) == expert_ids).sum().item()

    return {
        'loss': total_loss / max(n_samples, 1),
        'ce': total_ce / max(n_samples, 1),
        'aux': total_aux / max(n_samples, 1),
        'routing_acc': n_correct / max(n_samples, 1),
        'n': n_samples,
    }


def fit_router_head(
    model: VisionRouter,
    dataloader,
    device: str,
    epochs: int = 5,
    lr: float = 1e-3,
    alpha_aux: float = 0.05,
    head_only: bool = True,
    max_batches_per_epoch: int | None = None,
    ckpt_path: str | None = None,
):
    """
    Entrenamiento corto de la cabeza del router (Fase 19).
    LR por defecto 1e-3 alineado a Método A épocas 1-10 (consigna).
    """
    set_train_router_head_only(model, head_only=head_only)
    model.to(device)

    params = [p for p in model.router_head.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    use_cuda = str(device).startswith('cuda') and torch.cuda.is_available()
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda)

    for ep in range(1, epochs + 1):
        m = train_router_head_one_epoch(
            model, dataloader, optimizer, device,
            alpha_aux=alpha_aux, scaler=scaler if use_cuda else None,
            max_batches=max_batches_per_epoch,
        )
        print(f"[Fase19] Época {ep}/{epochs}  loss={m['loss']:.4f}  CE={m['ce']:.4f}  "
              f"L_aux={m['aux']:.4f}  RoutingAcc={m['routing_acc']:.4f}  n={m['n']}")

    if ckpt_path:
        os.makedirs(os.path.dirname(ckpt_path) or '.', exist_ok=True)
        torch.save({
            'router_head': model.router_head.state_dict(),
            'epoch': epochs,
            'alpha_aux': alpha_aux,
        }, ckpt_path)
        print(f"Checkpoint cabeza guardado: {ckpt_path}")

    return model


print('Funciones Fase 19 listas: set_train_router_head_only, fit_router_head.')

Funciones Fase 19 listas: set_train_router_head_only, fit_router_head.


In [91]:
def validate_cls_arrays(Z, y, name='Z', expected_dim=192):
    """Comprueba forma, NaN/Inf y estadísticas básicas de los CLS guardados."""
    Z = np.asarray(Z)
    y = np.asarray(y).ravel()
    print(f'=== {name} ===')
    print(f'  shape: {Z.shape}  dtype: {Z.dtype}')
    assert Z.ndim == 2, 'Z debe ser [N, d_model]'
    assert Z.shape[1] == expected_dim, f'Última dim debe ser {expected_dim}, got {Z.shape[1]}'
    assert len(y) == len(Z), 'y y Z deben tener el mismo N'

    n_bad = np.sum(~np.isfinite(Z))
    print(f'  NaN/Inf: {n_bad} / {Z.size}')
    if n_bad:
        print('  ⚠️ Corrige el extractor o datos antes del ablation.')

    print(f'  min/max: {np.nanmin(Z):.4f} / {np.nanmax(Z):.4f}')
    print(f'  mean/std (global): {np.mean(Z):.6f} / {np.std(Z):.6f}')

    for e in range(5):
        m = Z[y == e]
        if len(m) == 0:
            print(f'  experto {e}: sin muestras')
            continue
        norms = np.linalg.norm(m, axis=1)
        print(f'  experto {e}: n={len(m)}  ||CLS||_2 mean={norms.mean():.3f} std={norms.std():.3f}')
    print()
    return Z, y


def linear_probe_routing_accuracy(Z_train, y_train, Z_val, y_val, seed=42):
    """Clasificador lineal rápido: si >> 20%, los embeddings llevan señal de modalidad."""
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(max_iter=2000, random_state=seed, multi_class='multinomial')
    clf.fit(Z_train, y_train)
    acc = float(clf.score(Z_val, y_val))
    print(f'Linear probe (routing): val accuracy = {acc:.4f}  (chance ≈ 0.20)')
    return acc


# --- Ejemplo: cargar arrays guardados y validar (ajusta SAVE_DIR) ---
# SAVE_DIR = os.path.join(FEATURE_DIR, 'ablation_data_v1')
# Z_tr = np.load(os.path.join(SAVE_DIR, 'Z_train.npy'))
# y_tr = np.load(os.path.join(SAVE_DIR, 'y_train_expert.npy'))
# Z_va = np.load(os.path.join(SAVE_DIR, 'Z_val.npy'))
# y_va = np.load(os.path.join(SAVE_DIR, 'y_val_expert.npy'))
# validate_cls_arrays(Z_tr, y_tr, 'Z_train')
# validate_cls_arrays(Z_va, y_va, 'Z_val')
# linear_probe_routing_accuracy(Z_tr, y_tr, Z_va, y_va)

print('Funciones listas: validate_cls_arrays, linear_probe_routing_accuracy')

Funciones listas: validate_cls_arrays, linear_probe_routing_accuracy


In [92]:
def verify_cls_matches_model(
    model,
    preprocessor,
    file_paths,
    Z_reference_rows,
    device='cpu',
    rtol=1e-4,
    atol=1e-5,
):
    """
    Compara CLS recalculado en vivo con filas de Z ya guardadas (mismo orden de file_paths).
    Usa el mismo model.eval() y sin grad. Útil para depurar un subconjunto pequeño.
    """
    model.eval()
    model.to(device)
    diffs = []
    with torch.no_grad():
        for i, path in enumerate(file_paths):
            t = preprocessor(path).unsqueeze(0)
            # El forward del VisionRouter espera lista de tensores por muestra
            _, cls_live = model([t.squeeze(0)])
            v_live = cls_live.cpu().numpy().reshape(-1)
            v_ref = np.asarray(Z_reference_rows[i]).reshape(-1)
            err = np.max(np.abs(v_live - v_ref))
            diffs.append(err)
            ok = np.allclose(v_live, v_ref, rtol=rtol, atol=atol)
            print(f'  [{i}] {os.path.basename(path)}  max|Δ|={err:.2e}  ok={ok}')
    print(f'Máximo error global: {max(diffs):.2e}')
    return diffs


# Ejemplo (descomenta; necesitas mismos pesos que al extraer):
# paths = [all_files[0], all_files[100], ...][:5]
# Z_rows = Z_train[[0, 10, 20, 30, 40]]  # mismas filas que paths
# verify_cls_matches_model(router, preprocessor, paths, Z_rows, device=device)